# sklearn / TensorFlow — Course 2 (Advanced Learning Algorithms)

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler
from sklearn.metrics         import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.tree            import DecisionTreeClassifier, plot_tree
from sklearn.ensemble        import RandomForestClassifier
from xgboost                 import XGBClassifier

import tensorflow as tf
from tensorflow.keras              import Sequential
from tensorflow.keras.layers       import Dense
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers   import Adam
from tensorflow.keras.losses       import BinaryCrossentropy, SparseCategoricalCrossentropy

tf.random.set_seed(42)   # so results are reproducible, same idea as random_state in sklearn

## 2. Load Data

In [ ]:
df = pd.read_csv('your_file.csv')
df.head()

## 3. Prepare X and y — Split Data
Same 70 / 15 / 15 split idea as Course 1 — train, validation, test.
Neural networks need scaled features (like logistic regression did) — trees do NOT need scaling.

In [ ]:
feature_cols = ['col1', 'col2', 'col3']   # all input features
target_col   = 'target'                   # column with the class labels (0,1,2... or 0/1)

X = df[feature_cols].values
y = df[target_col].values

# 70% train, 15% val, 15% test — same pattern as course 1
X_train, X_tmp,  y_train, y_tmp  = train_test_split(X, y, test_size=0.30, random_state=42)
X_val,   X_test, y_val,   y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=42)

print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')

# scale ONLY for the neural network — fit on train, transform val/test with same scaler
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

---
# PART A — NEURAL NETWORK (Binary Classification)
Use this when target is 0/1 (e.g. malignant/benign, survived/not survived)

## 4a. Build the Architecture
Sequential just STACKS the layers — no training happens here, this is only the blueprint of the network.
Hidden layers use ReLU (fast, avoids flat-gradient problem). Output layer uses sigmoid since we want P(y=1|x).

In [ ]:
model_bin = Sequential([
    Dense(units=25, activation='relu', kernel_regularizer=l2(0.001)),  # hidden layer 1 - 25 neurons
    Dense(units=15, activation='relu', kernel_regularizer=l2(0.001)),  # hidden layer 2 - 15 neurons
    Dense(units=1,  activation='sigmoid')                              # output layer - probability of class 1
])

## 5a. Compile — define loss + optimizer
Adam is basically a smarter gradient descent — it adjusts the learning rate for each parameter on its own.

In [ ]:
model_bin.compile(
    loss=BinaryCrossentropy(),          # log loss, same idea as logistic regression cost function
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy']
)

## 6a. Fit — this is where the actual training (forward prop + backprop) happens

In [ ]:
history_bin = model_bin.fit(
    X_train_s, y_train,
    validation_data=(X_val_s, y_val),
    epochs=100,
    verbose=0     # set to 1 if you want to see progress per epoch
)

print('done training')

## 7a. Plot Training Curves (loss going down = model is learning)

In [ ]:
plt.plot(history_bin.history['loss'], label='train loss')
plt.plot(history_bin.history['val_loss'], label='val loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (Binary Cross-Entropy)')
plt.title('Neural Network — Training vs Validation Loss')
plt.legend()
plt.show()

# if val loss starts going UP while train loss keeps going down -> overfitting (high variance)
# if BOTH stay high and flat -> underfitting (high bias)

## 8a. Check Overfitting / Underfitting (Bias / Variance)

In [ ]:
y_pred_train_bin = (model_bin.predict(X_train_s) > 0.5).astype(int).flatten()
y_pred_val_bin   = (model_bin.predict(X_val_s)   > 0.5).astype(int).flatten()

train_acc = accuracy_score(y_train, y_pred_train_bin)
val_acc   = accuracy_score(y_val,   y_pred_val_bin)

print(f'Train Accuracy : {train_acc:.4f}')
print(f'Val   Accuracy : {val_acc:.4f}')
print()

if train_acc < 0.85 and val_acc < 0.85:
    print('-> HIGH BIAS (underfitting) - try a bigger network (more units/layers), train longer, or reduce l2 regularization')
elif train_acc - val_acc > 0.1:
    print('-> HIGH VARIANCE (overfitting) - add/increase l2 regularization, get more data, or use a smaller network')
else:
    print('-> Good fit')

## 9a. Confusion Matrix
Shows exactly where the model is getting confused (false positives vs false negatives)

In [ ]:
cm = confusion_matrix(y_val, y_pred_val_bin)
ConfusionMatrixDisplay(cm).plot()
plt.title('Neural Network — Confusion Matrix (Validation)')
plt.show()

---
# PART B — NEURAL NETWORK (Multiclass Classification)
Use this when target has more than 2 classes (e.g. digits 0-9, species A/B/C)
Target column should be integer labels: 0, 1, 2, 3...

## 4b. Prepare Multiclass Target
Reusing the same X, just showing this separately in case your multiclass target is a different column

In [ ]:
target_col_multi = 'target_multiclass'   # integer labels 0,1,2,...

y_multi = df[target_col_multi].values
num_classes = len(np.unique(y_multi))
print(f'Number of classes: {num_classes}')

X_train_m, X_tmp_m,  y_train_m, y_tmp_m  = train_test_split(X, y_multi, test_size=0.30, random_state=42)
X_val_m,   X_test_m, y_val_m,   y_test_m = train_test_split(X_tmp_m, y_tmp_m, test_size=0.50, random_state=42)

scaler_m = StandardScaler()
X_train_ms = scaler_m.fit_transform(X_train_m)
X_val_ms   = scaler_m.transform(X_val_m)
X_test_ms  = scaler_m.transform(X_test_m)

## 5b. Build Architecture — Softmax Output
IMPORTANT: output layer uses 'linear' NOT 'softmax' here — we output raw scores (logits)
and let from_logits=True handle softmax + cross-entropy together in one numerically stable step
(doing it separately causes round-off errors, especially in bigger networks)

In [ ]:
model_multi = Sequential([
    Dense(units=64, activation='relu', kernel_regularizer=l2(0.001)),
    Dense(units=32, activation='relu', kernel_regularizer=l2(0.001)),
    Dense(units=num_classes, activation='linear')   # linear output = logits, not probabilities yet
])

model_multi.compile(
    loss=SparseCategoricalCrossentropy(from_logits=True),   # use this since labels are plain integers (0,1,2..)
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy']
)

## 6b. Fit

In [ ]:
history_multi = model_multi.fit(
    X_train_ms, y_train_m,
    validation_data=(X_val_ms, y_val_m),
    epochs=100,
    verbose=0
)
print('done training')

## 7b. Plot Training Curves

In [ ]:
plt.plot(history_multi.history['loss'], label='train loss')
plt.plot(history_multi.history['val_loss'], label='val loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (Sparse Categorical Cross-Entropy)')
plt.title('Multiclass NN — Training vs Validation Loss')
plt.legend()
plt.show()

## 8b. Get Predictions
Since the model outputs raw logits, we manually apply softmax to turn them into probabilities,
then argmax to pick the class with the highest probability.

In [ ]:
logits_val = model_multi.predict(X_val_ms)
probs_val  = tf.nn.softmax(logits_val).numpy()      # now these sum to 1 per row
y_pred_val_multi = np.argmax(probs_val, axis=1)      # pick highest probability class

val_acc_multi = accuracy_score(y_val_m, y_pred_val_multi)
print(f'Val Accuracy : {val_acc_multi:.4f}')

## 9b. Check Bias / Variance

In [ ]:
logits_train = model_multi.predict(X_train_ms)
y_pred_train_multi = np.argmax(tf.nn.softmax(logits_train).numpy(), axis=1)
train_acc_multi = accuracy_score(y_train_m, y_pred_train_multi)

print(f'Train Accuracy : {train_acc_multi:.4f}')
print(f'Val   Accuracy : {val_acc_multi:.4f}')
print()

if train_acc_multi < 0.85 and val_acc_multi < 0.85:
    print('-> HIGH BIAS - bigger network, train longer, less regularization')
elif train_acc_multi - val_acc_multi > 0.1:
    print('-> HIGH VARIANCE - more regularization, more data, simpler network')
else:
    print('-> Good fit')

---
# PART C — DECISION TREE
Trees do NOT need feature scaling (splits are based on thresholds, not distances) — use the raw X_train/X_val

## 5c. Pipeline — Decision Tree
criterion='entropy' means it uses information gain to decide splits, same formula from the course

In [ ]:
tree = DecisionTreeClassifier(
    max_depth=6,             # stopping criteria - limits how deep the tree can grow
    criterion='entropy',     # use information gain (entropy reduction) to pick splits
    min_samples_split=10,    # another stopping criteria - won't split a node with fewer examples than this
    random_state=42
)

tree.fit(X_train, y_train)   # note: raw X_train, no scaling needed

y_pred_train_tree = tree.predict(X_train)
y_pred_val_tree   = tree.predict(X_val)

print(f'Train Accuracy : {accuracy_score(y_train, y_pred_train_tree):.4f}')
print(f'Val   Accuracy : {accuracy_score(y_val,   y_pred_val_tree):.4f}')

## 6c. Visualize the Tree (small max_depth is easier to read)

In [ ]:
plt.figure(figsize=(20, 10))
plot_tree(tree, feature_names=feature_cols, filled=True, fontsize=8, max_depth=3)
plt.title('Decision Tree Visualization (first 3 levels)')
plt.show()

## 7c. Check Overfitting / Underfitting + Tune max_depth

In [ ]:
train_acc_tree = accuracy_score(y_train, y_pred_train_tree)
val_acc_tree   = accuracy_score(y_val,   y_pred_val_tree)

if train_acc_tree < 0.85 and val_acc_tree < 0.85:
    print('-> HIGH BIAS (underfitting) - increase max_depth')
elif train_acc_tree - val_acc_tree > 0.1:
    print('-> HIGH VARIANCE (overfitting) - decrease max_depth, increase min_samples_split')
else:
    print('-> Good fit')

print()
print('=== Tuning max_depth ===')
for depth in [2, 4, 6, 8, 10, None]:
    t = DecisionTreeClassifier(max_depth=depth, criterion='entropy', random_state=42)
    t.fit(X_train, y_train)
    val_acc = accuracy_score(y_val, t.predict(X_val))
    print(f'max_depth={str(depth):<6}  val accuracy={val_acc:.4f}')
# pick the depth with the best val accuracy that isn't drastically overfitting on train

## 8c. Feature Importance

In [ ]:
importances = pd.Series(tree.feature_importances_, index=feature_cols).sort_values(ascending=False)
importances.plot(kind='bar')
plt.title('Decision Tree — Feature Importance')
plt.ylabel('Importance')
plt.show()
print(importances)

---
# PART D — RANDOM FOREST
Builds many trees (n_estimators) - each on a random bootstrap sample of data + random subset of features at each split.
Then all trees vote on the final prediction. This reduces variance compared to a single tree.

## 5d. Pipeline — Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,     # number of trees in the forest
    max_depth=6,
    max_features='sqrt',  # random feature subset size considered at each split
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_train_rf = rf.predict(X_train)
y_pred_val_rf   = rf.predict(X_val)

print(f'Train Accuracy : {accuracy_score(y_train, y_pred_train_rf):.4f}')
print(f'Val   Accuracy : {accuracy_score(y_val,   y_pred_val_rf):.4f}')

## 6d. Tune n_estimators and max_depth

In [ ]:
print('=== Tuning Random Forest ===')
for n in [50, 100, 200, 300]:
    for depth in [4, 6, 8]:
        m = RandomForestClassifier(n_estimators=n, max_depth=depth, random_state=42)
        m.fit(X_train, y_train)
        val_acc = accuracy_score(y_val, m.predict(X_val))
        print(f'n_estimators={n:<5} max_depth={depth:<4} val accuracy={val_acc:.4f}')

## 7d. Feature Importance — Random Forest usually gives more reliable importances than a single tree

In [ ]:
importances_rf = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
importances_rf.plot(kind='bar')
plt.title('Random Forest — Feature Importance')
plt.ylabel('Importance')
plt.show()

---
# PART E — XGBOOST (Boosted Trees)
Unlike Random Forest (trees built independently/in parallel), boosting builds trees SEQUENTIALLY -
each new tree focuses on fixing the mistakes (misclassified/high-error examples) of the previous trees.

## 5e. Pipeline — XGBoost

In [ ]:
xgb = XGBClassifier(
    n_estimators=200,       # number of sequential trees
    max_depth=6,
    learning_rate=0.1,      # how much each new tree corrects previous mistakes (shrinkage)
    eval_metric='logloss',  # use 'mlogloss' instead if multiclass
    random_state=42
)

xgb.fit(X_train, y_train)

y_pred_train_xgb = xgb.predict(X_train)
y_pred_val_xgb   = xgb.predict(X_val)

print(f'Train Accuracy : {accuracy_score(y_train, y_pred_train_xgb):.4f}')
print(f'Val   Accuracy : {accuracy_score(y_val,   y_pred_val_xgb):.4f}')

## 6e. Early Stopping — stop adding trees once val performance stops improving (prevents overfitting)

In [ ]:
xgb_es = XGBClassifier(
    n_estimators=1000,          # set high, early stopping will cut it short
    max_depth=6,
    learning_rate=0.1,
    eval_metric='logloss',
    early_stopping_rounds=20,   # stop if val score doesn't improve for 20 rounds
    random_state=42
)

xgb_es.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print(f'Best iteration: {xgb_es.best_iteration}')
print(f'Val Accuracy  : {accuracy_score(y_val, xgb_es.predict(X_val)):.4f}')

## 7e. Tune learning_rate and max_depth

In [ ]:
print('=== Tuning XGBoost ===')
for lr in [0.01, 0.1, 0.3]:
    for depth in [3, 6, 9]:
        m = XGBClassifier(n_estimators=200, max_depth=depth, learning_rate=lr,
                           eval_metric='logloss', random_state=42)
        m.fit(X_train, y_train)
        val_acc = accuracy_score(y_val, m.predict(X_val))
        print(f'lr={lr:<6} max_depth={depth:<4} val accuracy={val_acc:.4f}')

## 8e. Feature Importance — XGBoost

In [ ]:
importances_xgb = pd.Series(xgb.feature_importances_, index=feature_cols).sort_values(ascending=False)
importances_xgb.plot(kind='bar')
plt.title('XGBoost — Feature Importance')
plt.ylabel('Importance')
plt.show()

---
# PART F — Compare All Models & Final Test Evaluation
Only touch the test set ONCE, after picking your best model from validation results above.

## 9. Compare Validation Accuracy Across All Models

In [ ]:
results = {
    'Neural Network (binary)': accuracy_score(y_val, y_pred_val_bin),
    'Decision Tree':           accuracy_score(y_val, y_pred_val_tree),
    'Random Forest':           accuracy_score(y_val, y_pred_val_rf),
    'XGBoost':                 accuracy_score(y_val, y_pred_val_xgb),
}

results_df = pd.Series(results).sort_values(ascending=False)
print(results_df)

results_df.plot(kind='barh')
plt.xlabel('Validation Accuracy')
plt.title('Model Comparison')
plt.show()

# whichever has the best val accuracy (without a huge train-val gap) is your pick for final testing

## 10. Final Evaluation on Test Set
Run this ONCE, only for the model you picked above

In [ ]:
# example: if XGBoost was the winner
test_acc_final = accuracy_score(y_test, xgb.predict(X_test))
print(f'Final Test Accuracy : {test_acc_final:.4f}')

# test accuracy close to val accuracy -> model generalizes well
# test accuracy much worse than val -> you over-tuned on the validation set